# ORION P1–P15 evidence atlas

## Scope and authority

This notebook is a **receipt browser**, not a new experiment and not a publication-readiness certificate. It visualizes normalized fields copied from repository evidence. A local replay, finite witness, hash, or passing harness can establish only its registered bounded scope. It does **not** establish external independence, broad generalization, superiority, novelty, or top-tier readiness. `FAIL`, `UNKNOWN`, `CANNOT_CHECK`, `GATE_NOT_MET`, `NOT_AUTHORITY`, and not-executed outcomes remain visible.

The notebook reads `visualization/data/derived/atlas.json`; it never replaces the cited source receipt.


## Theory and methodology

ORION treats epistemic state and claim authority as typed, revisable objects. A compact reading aid is

$$S_t=(K_t,W_t,M_t), \qquad S_{t+1}=\mathcal{T}(S_t,e_t),$$

where $K$ is current knowledge, $W$ records unresolved/unknown material, $M$ records methods or mechanics, and $e_t$ is new evidence. The visual atlas adds no new transition: it projects already committed receipts.

The non-escalation rule is represented schematically as

$$\operatorname{claim}(r)\;\preceq\;\operatorname{authority}(r)\;\preceq\;\operatorname{scope}(r).$$

`\preceq` means “no broader than,” not a numeric score. The categorical matrix below shows **membership in the atlas's recorded status classes**; exact terminal strings remain visible in the table rather than becoming unreadable axis labels. Its cells are not ordered quality grades.


In [ ]:
from pathlib import Path
import json
import sys


def find_visualization_root(start=Path.cwd()):
    """Find visualization/ whether Jupyter starts at the repo root or notebooks/."""
    start = start.resolve()
    candidates = [start / "visualization", start, *start.parents]
    for candidate in candidates:
        if candidate.name == "visualization" and (candidate / "data" / "derived" / "atlas.json").exists():
            return candidate
        nested = candidate / "visualization"
        if (nested / "data" / "derived" / "atlas.json").exists():
            return nested
    raise FileNotFoundError(
        "Could not find visualization/data/derived/atlas.json. "
        "Build the atlas from the repository root first."
    )


VIS_ROOT = find_visualization_root()
sys.path.insert(0, str(VIS_ROOT / "src"))
ATLAS_PATH = VIS_ROOT / "data" / "derived" / "atlas.json"
atlas = json.loads(ATLAS_PATH.read_text(encoding="utf-8"))


def as_rows(value):
    """Return normalized records without changing their scientific values."""
    if isinstance(value, list):
        return [row for row in value if isinstance(row, dict)]
    if isinstance(value, dict):
        return [row for row in value.values() if isinstance(row, dict)]
    return []


def first(row, *keys, default=None):
    for key in keys:
        if key in row and row[key] is not None:
            return row[key]
    return default


def paper_id(row):
    raw = str(first(row, "paper_id", "paper", "id", default="UNSCOPED"))
    return raw.replace("ORION-", "")


def exact_status(row):
    return str(first(row, "terminal", "status", "result_state", "authority", default="UNSPECIFIED"))


def numeric_value(row):
    value = first(row, "value", "observed", "count", default=None)
    return float(value) if isinstance(value, (int, float)) and not isinstance(value, bool) else None


paper_states = as_rows(atlas.get("paper_states", []))
metrics_by_paper = atlas.get("metrics", {})
metrics = as_rows(atlas.get("metric_records", []))
anomalies = as_rows(atlas.get("anomalies", []))
sources = as_rows(atlas.get("sources", []))
des_execution = as_rows(atlas.get("des_execution", []))
framework_mechanics = atlas.get("framework_mechanics", {})

print(f"Atlas: {ATLAS_PATH}")
print(
    f"Loaded {len(paper_states)} paper states, {len(metrics)} metrics, "
    f"{len(anomalies)} anomalies, {len(des_execution)} frozen DES rows and "
    f"{len(sources)} sources."
)


In [ ]:
import matplotlib.pyplot as plt
import textwrap
from matplotlib.colors import ListedColormap  # noqa: F401 -- used by heatmap notebooks

plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "axes.grid": True,
    "grid.alpha": 0.20,
    "font.size": 10,
})

STATE_COLORS = {
    "PASS": "#2e7d32",
    "SUPPORTED": "#2e7d32",
    "FAIL": "#c62828",
    "GATE_NOT_MET": "#c62828",
    "CANNOT_CHECK": "#ef6c00",
    "UNKNOWN": "#6a1b9a",
    "NOT_AUTHORITY": "#455a64",
    "NOT_EXECUTED": "#757575",
}


def state_color(text):
    upper = str(text).upper()
    for token, color in STATE_COLORS.items():
        if token in upper:
            return color
    return "#1565c0"


def human_label(value, width=18):
    # Wrap machine identifiers without changing canonical capitalization.
    cleaned = str(value).replace("_", " ").replace(":", " — ")
    return "\n".join(textwrap.wrap(cleaned, width=width, break_long_words=False))



def print_records(rows, fields, limit=30):
    """Small dependency-free table for exact atlas fields."""
    rows = list(rows)
    if not rows:
        print("No records match the current display selectors.")
        return
    widths = {
        field: min(
            48,
            max(len(field), *(len(str(first(row, field, default=""))) for row in rows[:limit])),
        )
        for field in fields
    }
    print(" | ".join(field.ljust(widths[field]) for field in fields))
    print("-+-".join("-" * widths[field] for field in fields))
    for row in rows[:limit]:
        print(" | ".join(str(first(row, field, default=""))[: widths[field]].ljust(widths[field]) for field in fields))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more record(s); change DISPLAY_LIMIT to inspect them.")


## Editable display selectors

Edit and rerun the next cell. Selectors only change the view; they never change the atlas or erase the full anomaly count printed later.


In [ ]:
PAPERS = [f"P{i}" for i in range(1, 16)]
STATUS_CONTAINS = None      # e.g. "CANNOT_CHECK"; None shows every exact state
DISPLAY_LIMIT = 30


## Results: recorded status-class map

Each filled cell says only that the recorded status class occurs for that paper in the normalized atlas. Exact terminals are retained in the table below. Multiple cells in a row are expected when a paper has multiple scoped results.


In [ ]:
selected_states = [
    row for row in paper_states
    if paper_id(row) in PAPERS
    and (STATUS_CONTAINS is None or STATUS_CONTAINS.upper() in exact_status(row).upper())
]

status_classes = sorted({str(first(row, "status", default="UNSPECIFIED")) for row in selected_states})
matrix = [
    [int(any(paper_id(row) == pid and str(first(row, "status", default="UNSPECIFIED")) == state for row in selected_states))
     for state in status_classes]
    for pid in PAPERS
]

fig, ax = plt.subplots(figsize=(max(9, 1.15 * len(status_classes)), 7))
if status_classes:
    image = ax.imshow(matrix, aspect="auto", cmap=ListedColormap(["#f5f5f5", "#1565c0"]), vmin=0, vmax=1)
    ax.set_xticks(range(len(status_classes)), [human_label(state, 16) for state in status_classes])
    ax.set_yticks(range(len(PAPERS)), PAPERS)
    ax.grid(False)
    ax.set_title("Recorded status-class membership (binary display, not a quality scale)")
    for i, row in enumerate(matrix):
        for j, present in enumerate(row):
            if present:
                ax.text(j, i, "●", ha="center", va="center", color="white", fontsize=8)
else:
    ax.text(0.5, 0.5, "No states match the selectors", ha="center", va="center")
    ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
print_records(
    selected_states,
    ["paper_id", "title", "status", "terminal", "authority", "claim_ceiling"],
    DISPLAY_LIMIT,
)
print(f"\nUnfiltered anomaly records retained in atlas: {len(anomalies)}")


## Discussion and anomaly reading

Heterogeneous terminal strings are intentional: ORION separates execution, scientific validity, and claim authority. A green local result beside `CANNOT_CHECK` is not contradictory when the former concerns a bounded replay and the latter concerns external or prospective authority. Review the anomaly notebook before interpreting any isolated metric.

## Claim ceiling

This view can support statements about what the cited repository receipts record. It cannot independently validate those receipts or convert local/bounded evidence into submission readiness. See `visualization/reports/CLAIM_CEILINGS.md`.
